# LC #155 — Min Stack
**Category:** Stack with Auxiliary State | **Difficulty:** Medium | **Day 3**

---

<div style="padding:15px;border-left:8px solid #f7971e;background:#fff8e1;border-radius:4px;">
<strong>The Problem:</strong> Design a stack supporting <code>push</code>, <code>pop</code>,
<code>top</code>, and <code>getMin()</code> — all in O(1) time.
</div>

**Interface:**
```
MinStack ms = MinStack()
ms.push(-2)
ms.push(0)
ms.push(-3)
ms.getMin()  → -3
ms.pop()
ms.top()     → 0
ms.getMin()  → -2
```

**Core Insight:** A parallel `min_stack` tracks the running minimum at every state. When you pop the main stack, pop min_stack too — the previous minimum is automatically exposed.

## Mental Models

**1. Two Synchronized Stacks**
`stack` holds actual values. `min_stack` holds the current minimum at each corresponding stack state. They stay in sync: push to both, pop from both simultaneously.

**2. Why Parallel, Not Just One Value?**
If you only stored one minimum variable and then popped it, you'd lose the *previous* minimum. The parallel stack remembers the minimum at *every* state the stack has been in.

**3. The Invariant**
`min_stack[-1]` is always the minimum of all elements currently in `stack`. This invariant is maintained by pushing `min(val, min_stack[-1])` on every push.

In [ ]:
# Alternative: single stack storing (value, min_at_this_point) tuples
# Same complexity but cleaner mental model for some

class MinStackTuple:
    def __init__(self):
        self.stack = []   # each entry: (value, min_so_far)

    def push(self, val: int) -> None:
        current_min = min(val, self.stack[-1][1]) if self.stack else val
        self.stack.append((val, current_min))

    def pop(self) -> None:
        self.stack.pop()

    def top(self) -> int:
        return self.stack[-1][0]

    def getMin(self) -> int:
        return self.stack[-1][1]

# Test
ms = MinStackTuple()
ms.push(-2); ms.push(0); ms.push(-3)
print(f"getMin: {ms.getMin()}")   # -3
ms.pop()
print(f"top:    {ms.top()}")      # 0
print(f"getMin: {ms.getMin()}")   # -2

## State Trace

Operations: `push(-2)`, `push(0)`, `push(-3)`, `pop()`, `top()`, `getMin()`

| Operation | stack | min_stack | Notes |
|-----------|-------|-----------|-------|
| push(-2) | [-2] | [-2] | first element, min = itself |
| push(0) | [-2, 0] | [-2, -2] | 0 > -2, min stays -2 |
| push(-3) | [-2, 0, -3] | [-2, -2, -3] | -3 < -2, new min |
| getMin() | — | — | min_stack[-1] = **-3** |
| pop() | [-2, 0] | [-2, -2] | both stacks sync-popped |
| top() | — | — | stack[-1] = **0** |
| getMin() | — | — | min_stack[-1] = **-2** ✓ |

After popping -3, the previous minimum (-2) is automatically the top of min_stack.

In [ ]:
# Optimal — O(1) all operations, O(n) space
# Two synchronized stacks.

class MinStack:
    def __init__(self):
        self.stack = []
        self.min_stack = []   # min_stack[-1] = current minimum

    def push(self, val: int) -> None:
        self.stack.append(val)
        # Current min is either new val or existing min — whichever is smaller
        if self.min_stack:
            self.min_stack.append(min(val, self.min_stack[-1]))
        else:
            self.min_stack.append(val)

    def pop(self) -> None:
        self.stack.pop()
        self.min_stack.pop()    # sync: both stacks stay the same length

    def top(self) -> int:
        return self.stack[-1]

    def getMin(self) -> int:
        return self.min_stack[-1]

# Test
ms = MinStack()
ms.push(-2); ms.push(0); ms.push(-3)
print(f"getMin after push(-2,0,-3): {ms.getMin()}")   # -3
ms.pop()
print(f"top after pop:              {ms.top()}")       # 0
print(f"getMin after pop:           {ms.getMin()}")    # -2

## Complexity Analysis

| Operation | Time | Space |
|-----------|------|-------|
| push | O(1) | O(n) total |
| pop | O(1) | |
| top | O(1) | |
| getMin | O(1) | |

**Space:** O(n) — both stacks combined hold 2n elements in the worst case. This is the trade-off for O(1) getMin.

**Alternative approaches:**
- Store `(val, min)` pairs in single stack — same complexity, different ergonomics
- Store only the deltas (space optimization for large min values) — same O(1) but more complex

## Interview Q&A

**Q: What's the space trade-off?**
A: Double the memory — O(n) for both stacks combined. In exchange, getMin() is O(1) instead of O(n) (scanning the full stack). Classic space-for-time trade.

**Q: How would you track the maximum instead of the minimum?**
A: Identical pattern with a `max_stack`. Push `max(val, max_stack[-1])` instead of `min(...)`.

**Q: Why does this need two stacks instead of one variable?**
A: A single variable loses the history. When you pop the current minimum, you need to know what the previous minimum was. The parallel stack preserves the minimum at every point in history.

**Q: What if you popped from an empty stack?**
A: The problem guarantees valid calls. In production code, you'd add an assertion or raise an exception on empty pop/top/getMin.

## The Citi Angle

**The "minimum with history" pattern** shows up in sliding-window baseline tracking:

```python
# Track running minimum CPU baseline with the ability to "roll back"
# (e.g., when a server is decommissioned, restore the previous baseline)

class BaselineTracker:
    """Tracks minimum CPU baseline — supports undo/rollback."""
    def __init__(self):
        self._readings = []
        self._min_stack = []

    def record(self, cpu: float):
        self._readings.append(cpu)
        prev_min = self._min_stack[-1] if self._min_stack else float('inf')
        self._min_stack.append(min(cpu, prev_min))

    def rollback(self):
        """Remove last reading (server decommissioned)."""
        if self._readings:
            self._readings.pop()
            self._min_stack.pop()

    def baseline(self) -> float:
        return self._min_stack[-1] if self._min_stack else float('inf')

tracker = BaselineTracker()
for cpu in [72.5, 68.1, 91.3, 45.2, 88.7]:
    tracker.record(cpu)
    print(f"CPU: {cpu:.1f}  Baseline min: {tracker.baseline():.1f}")
tracker.rollback()  # remove 88.7
print(f"After rollback, baseline: {tracker.baseline():.1f}")
```

**Interview tie-in:** "Min Stack is the building block for any system that needs O(1) minimum queries with rollback — capacity baseline tracking, undo operations in config management."

## Summary

| | Value |
|--|--|
| **Pattern** | Two synchronized stacks (main + min_stack) |
| **All operations** | O(1) |
| **Space** | O(n) — 2n total across both stacks |
| **Invariant** | `min_stack[-1]` always = current minimum of all stack elements |
| **Say in interview** | "Parallel min_stack: push min(val, current_min), pop both simultaneously. Invariant: min_stack[-1] is always the current minimum." |